# Подготовка к работе с данными

Первый этап проекта «Дашборд конверсий». Изучаем две учебные выгрузки образовательной платформы: посещения и первые регистрации.

Источники: [визиты](https://drive.google.com/file/d/1QosQQ4RRNR9rkL4t7sB707h2Uy0XfYJe/view), [регистрации](https://drive.google.com/file/d/1AeQz0kaSgz0lxYSDtuNm36muhy5fRCzZ/view).

Выполните ячейки сверху вниз из корня репозитория. При первом запуске нужен интернет. CSV сохраняются в `data/`, повторный запуск читает локальные копии. Чтобы обновить выгрузки, удалите эти CSV и повторите выполнение.


In [1]:
import pandas as pd
import requests

print(f"pandas: {pd.__version__}; requests: {requests.__version__}")

pandas: 3.0.5; requests: 2.34.2


## Загрузка CSV

Сохраняем исходные файлы без преобразований. Проверяем HTTP-ответ и заголовок CSV, чтобы не принять страницу Google Drive за данные. Все поля сначала читаем как строки: идентификаторы не являются числовыми показателями.

In [2]:
def load_csv(file_id, filename, expected_columns):
    try:
        with open(filename, "rb") as source:
            content = source.read()
    except FileNotFoundError:
        response = requests.get(
            "https://drive.google.com/uc",
            params={"export": "download", "id": file_id},
            timeout=60,
        )
        response.raise_for_status()
        content = response.content
        header = content.decode("utf-8-sig").splitlines()[0].split(",")
        if header != expected_columns:
            raise ValueError(f"{filename}: вместо ожидаемого CSV получен другой ответ")
        with open(filename, "wb") as target:
            target.write(content)

    frame = pd.read_csv(filename, dtype="string")
    if frame.columns.tolist() != expected_columns:
        raise ValueError(f"{filename}: неожиданный набор столбцов")
    return frame

visits = load_csv(
    "1QosQQ4RRNR9rkL4t7sB707h2Uy0XfYJe",
    "data/visits.csv",
    ["uuid", "platform", "user_agent", "date"],
)
registrations = load_csv(
    "1AeQz0kaSgz0lxYSDtuNm36muhy5fRCzZ",
    "data/registrations.csv",
    ["date", "user_id", "email", "platform", "registration_type"],
)
print("Визиты:", visits.shape)
print("Регистрации:", registrations.shape)

Визиты: (1000, 4)
Регистрации: (1000, 5)


## Первые строки и структура

In [3]:
visits.head()

,uuid,platform,user_agent,date
0,1de9ea66-70d3-4a1f-8735-df5ef7697fb9,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_11_2...,2023-03-01T13:29:22
1,f149f542-e935-4870-9734-6b4501eaf614,web,Mozilla/5.0 (X11; CrOS x86_64 8172.45.0) Apple...,2023-03-01T16:44:28
2,f149f542-e935-4870-9734-6b4501eaf614,web,Mozilla/5.0 (X11; CrOS x86_64 8172.45.0) Apple...,2023-03-06T06:12:36
3,08f0ebd4-950c-4dd9-8e97-b5bdf073eed1,web,Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:109...,2023-03-01T20:16:37
4,08f0ebd4-950c-4dd9-8e97-b5bdf073eed1,web,Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:109...,2023-03-05T17:42:47


In [4]:
registrations.head()

,date,user_id,email,platform,registration_type
0,2023-03-01T00:25:39,8838849,joseph95@example.org,web,google
1,2023-03-01T14:53:01,8741065,janetsuarez@example.net,web,yandex
2,2023-03-01T14:27:36,1866654,robert67@example.org,web,google
3,2023-03-01T02:42:34,1577584,elam@example.net,web,apple
4,2023-03-01T10:27:14,4765395,stephanie68@example.net,web,yandex


In [5]:
visits.info()
registrations.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   uuid        1000 non-null   string
 1   platform    1000 non-null   string
 2   user_agent  1000 non-null   string
 3   date        1000 non-null   string
dtypes: string(4)
memory usage: 31.4 KB
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   date               1000 non-null   string
 1   user_id            1000 non-null   string
 2   email              1000 non-null   string
 3   platform           1000 non-null   string
 4   registration_type  1000 non-null   string
dtypes: string(5)
memory usage: 39.2 KB


## Предварительный анализ через describe

`describe(include="all")` показывает все столбцы, включая текстовые: `count` — число непустых значений, `unique` — число различных значений, `top` — одно из самых частых значений, `freq` — его частота. При одинаковой частоте `top` не означает единственного лидера.

In [6]:
visits.describe(include="all")

,uuid,platform,user_agent,date
count,1000,1000,1000,1000
unique,519,3,28,996
top,251a0926-ece3-4d77-aa42-ab569fdf9fe2,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,2023-03-01T08:01:45
freq,4,954,71,2


In [7]:
registrations.describe(include="all")

,date,user_id,email,platform,registration_type
count,1000,1000,1000,1000,1000
unique,1000,1000,997,3,4
top,2023-03-01T00:25:39,8838849,zanderson@example.org,android,email
freq,1,1,2,517,446


## Пропуски, дубликаты и категории

In [8]:
pd.DataFrame({
    "visits_missing": visits.isna().sum(),
    "registrations_missing": registrations.isna().sum(),
}).astype("Int64")

,visits_missing,registrations_missing
date,0,0
email,<NA>,0
platform,0,0
registration_type,<NA>,0
user_agent,0,<NA>
user_id,<NA>,0
uuid,0,<NA>


В объединённой таблице `<NA>` означает, что столбца нет в соответствующей выгрузке.

In [9]:
pd.DataFrame({
    "rows": [len(visits), len(registrations)],
    "full_duplicates": [visits.duplicated().sum(), registrations.duplicated().sum()],
    "unique_ids": [visits["uuid"].nunique(), registrations["user_id"].nunique()],
}, index=["visits", "registrations"])

,rows,full_duplicates,unique_ids
visits,1000,0,519
registrations,1000,0,1000


In [10]:
pd.concat({
    "visits": visits["platform"].value_counts(),
    "registrations": registrations["platform"].value_counts(),
}, axis=1).fillna(0).astype(int)

,visits,registrations
platform,,
web,954,265
android,27,517
ios,19,218


In [11]:
registrations["registration_type"].value_counts()

registration_type
email     446
google    303
apple     178
yandex     73
Name: count, dtype: Int64

In [12]:
print("Уникальных email:", registrations["email"].nunique())
print("Повторов email сверх первого:", registrations["email"].duplicated().sum())

Уникальных email: 997
Повторов email сверх первого: 3


## Даты и охват выгрузок

Создаём отдельные Series дат для анализа, не меняя исходные DataFrame. Ошибки разбора становятся `NaT` и учитываются явно. Часовой пояс в источниках не указан.

In [13]:
visit_dates = pd.to_datetime(visits["date"], format="%Y-%m-%dT%H:%M:%S", errors="coerce")
registration_dates = pd.to_datetime(registrations["date"], format="%Y-%m-%dT%H:%M:%S", errors="coerce")

pd.DataFrame({
    "start": [visit_dates.min(), registration_dates.min()],
    "end": [visit_dates.max(), registration_dates.max()],
    "invalid_or_missing_dates": [visit_dates.isna().sum(), registration_dates.isna().sum()],
}, index=["visits", "registrations"])

,start,end,invalid_or_missing_dates
visits,2023-03-01 00:05:35,2023-03-07 23:05:08,0
registrations,2023-03-01 00:12:22,2023-03-05 22:04:01,0


## Выводы

- В выгрузке посещений 1000 строк и 4 столбца, в регистрациях — 1000 строк и 5 столбцов. Пропусков и полных дубликатов нет. Все даты разобраны успешно.
- У посещений 519 уникальных `uuid`: один посетитель мог заходить несколько раз. Повторение `uuid` само по себе не является ошибкой.
- Среди посещений 954 относятся к `web`, 27 — к `android`, 19 — к `ios`. Значение `bot` в поле `platform` отсутствует. Это не доказывает отсутствие ботов среди посещений: для этого потребуется отдельно изучить `user_agent`.
- Все 1000 `user_id` уникальны, но email только 997: есть 3 повторения сверх первого вхождения. Удалять такие регистрации без дополнительных правил пока нельзя.
- Среди регистраций лидирует `android` (517), затем `web` (265) и `ios` (218). Самый частый тип регистрации — `email` (446).
- Визиты охватывают 1–7 марта 2023 года, регистрации — 1–5 марта. Эти учебные выборки не дают основания считать конверсию как 1000 / 1000: периоды различаются, полнота выгрузок неизвестна, общего ключа для связи `uuid` и `user_id` нет.

На этом этапе данные загружены и изучены. Очистку и расчёт конверсии выполняем после уточнения правил следующих этапов.


# Запросы к API

Получаем посещения и регистрации за период **2023-03-01 → 2023-09-01**.
Используем маршруты `/visits` и `/registrations` сервиса
[data-charts-api.hexlet.app](https://data-charts-api.hexlet.app),
параметры `begin` и `end` передаём через `requests.get(..., params=...)`.

Каждый запуск этого раздела заново обращается к API и требует интернета.
Полные ответы загружаются в `api_visits` и `api_registrations`; в notebook
выводим только первые строки и сводку, чтобы не сохранять сотни тысяч строк в выводах.
Данные первого этапа остаются в переменных `visits` и `registrations`.


In [14]:
API_BASE_URL = "https://data-charts-api.hexlet.app"
API_PERIOD = {"begin": "2023-03-01", "end": "2023-09-01"}


def fetch_api_data(endpoint, expected_columns):
    response = requests.get(
        f"{API_BASE_URL}/{endpoint}",
        params=API_PERIOD,
        timeout=120,
    )
    response.raise_for_status()
    records = response.json()
    if not isinstance(records, list):
        raise ValueError(f"{endpoint}: ожидался JSON-массив записей")
    if not all(isinstance(record, dict) for record in records):
        raise ValueError(f"{endpoint}: каждая запись должна быть JSON-объектом")
    if not all(set(expected_columns).issubset(record) for record in records):
        raise ValueError(f"{endpoint}: в записях отсутствуют обязательные поля")

    frame = pd.DataFrame(records, columns=expected_columns).astype("string")
    print(f"{endpoint}: HTTP {response.status_code}, строк: {len(frame):,}")
    return frame


api_visits = fetch_api_data(
    "visits", ["visit_id", "platform", "user_agent", "datetime"]
)
api_registrations = fetch_api_data(
    "registrations",
    ["user_id", "email", "platform", "registration_type", "datetime"],
)

visits: HTTP 200, строк: 263,459


registrations: HTTP 200, строк: 21,836


In [15]:
api_visits.head()

,visit_id,platform,user_agent,datetime
0,a6fc64df-f1b9-4f6d-8d23-08a23d6946d0,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,2023-08-07T09:07:37
1,a6fc64df-f1b9-4f6d-8d23-08a23d6946d0,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,2023-08-04T09:31:21
2,a7abbf4b-1c20-4d00-989e-acd3ba8680e1,web,Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebKi...,2023-08-07T05:59:04
3,a7abbf4b-1c20-4d00-989e-acd3ba8680e1,web,Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebKi...,2023-08-06T00:50:59
4,a7abbf4b-1c20-4d00-989e-acd3ba8680e1,web,Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebKi...,2023-08-05T11:15:18


In [16]:
api_registrations.head()

,user_id,email,platform,registration_type,datetime
0,2e0f6bb8-b029-4f45-a786-2b53990d37f1,ebyrd@example.org,web,google,2023-03-01T07:40:13
1,f007f97c-9d8b-48b5-af08-119bb8f6d9b6,knightgerald@example.org,web,email,2023-03-01T13:14:00
2,24ff46ae-32b3-4a74-8f27-7cf0b8f32f15,cherylthompson@example.com,web,apple,2023-03-01T03:05:50
3,3e9914e1-5d73-4c23-b25d-b59a3aeb2b60,halldavid@example.org,web,email,2023-03-01T00:04:47
4,27f875fc-f8ce-4aeb-8722-0ecb283d0760,denise86@example.net,web,google,2023-03-01T18:31:52


## Проверка полученных данных

Сравниваем размеры таблиц и фактические диапазоны дат. Исходное поле `datetime`
сохраняем строковым, для проверки разбираем его в отдельную Series.
Параметр `end` передаём ровно как в задании, без добавления суток.
Сводка показывает фактический охват ответа; включительность границы API
по одному ответу не определяем.

In [17]:
api_summary_rows = []
for name, frame in {"visits": api_visits, "registrations": api_registrations}.items():
    dates = pd.to_datetime(frame["datetime"], format="ISO8601", errors="coerce")
    api_summary_rows.append({
        "dataset": name,
        "rows": len(frame),
        "columns": len(frame.columns),
        "first_datetime": dates.min(),
        "last_datetime": dates.max(),
        "invalid_or_missing_datetime": int(dates.isna().sum()),
        "outside_requested_dates": int((
            (dates.dt.strftime("%Y-%m-%d") < API_PERIOD["begin"])
            | (dates.dt.strftime("%Y-%m-%d") > API_PERIOD["end"])
        ).sum()),
    })

api_summary = pd.DataFrame(api_summary_rows).set_index("dataset")
api_summary

,rows,columns,first_datetime,last_datetime,invalid_or_missing_datetime,outside_requested_dates
dataset,,,,,,
visits,263459,4,2023-03-01 00:00:43,2023-08-31 23:52:57,0,0
registrations,21836,5,2023-03-01 00:04:47,2023-08-31 23:43:26,0,0


Данные API получены в отдельные DataFrame. В ответе посещений используются
поля `visit_id` и `datetime`, тогда как в CSV — `uuid` и `date`.
В регистрациях API поле времени также называется `datetime`.
На этом этапе не объединяем источники и не переименовываем поля.
